## Cross Validation

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [5]:
#Load dataset
data=load_wine()

X=pd.DataFrame(data.data, columns=data.feature_names) #type:ignore
y=pd.Series(data.target) #type:ignore

#Display info
print(X.head())
print("Shape:", X.shape)

print("\nSummary statistics:")
print(X.describe())

print("\nClass distribution:")
print(y.value_counts())

   alcohol  malic_acid   ash  ...   hue  od280/od315_of_diluted_wines  proline
0    14.23        1.71  2.43  ...  1.04                          3.92   1065.0
1    13.20        1.78  2.14  ...  1.05                          3.40   1050.0
2    13.16        2.36  2.67  ...  1.03                          3.17   1185.0
3    14.37        1.95  2.50  ...  0.86                          3.45   1480.0
4    13.24        2.59  2.87  ...  1.04                          2.93    735.0

[5 rows x 13 columns]
Shape: (178, 13)

Summary statistics:
          alcohol  malic_acid  ...  od280/od315_of_diluted_wines      proline
count  178.000000  178.000000  ...                    178.000000   178.000000
mean    13.000618    2.336348  ...                      2.611685   746.893258
std      0.811827    1.117146  ...                      0.709990   314.907474
min     11.030000    0.740000  ...                      1.270000   278.000000
25%     12.362500    1.602500  ...                      1.937500   500.5000

## Baseline Model (Train-Test Split)

In [6]:
#Split
X_train,X_test, y_train, y_test=train_test_split(
    X, y, test_size=0.2, random_state=42
)

#Model
model=LogisticRegression(max_iter=5000)

#Train
model.fit(X_train, y_train)

#Predict
y_pred=model.predict(X_test)

#Accuracy
acc=accuracy_score(y_test, y_pred)
print("Baseline Accuracy:", acc)

Baseline Accuracy: 1.0


## K-Fold Cross Validation

In [10]:
kf=KFold(n_splits=5, shuffle=True, random_state=42)

model=LogisticRegression(max_iter=10000)

scores=cross_val_score(model, X, y, cv=kf, scoring="accuracy") #One of the important function

print("Fold scores:", scores)
print("Mean:", scores.mean())
print("STD:", scores.std())

Fold scores: [1.         0.91666667 0.94444444 0.97142857 0.97142857]
Mean: 0.9607936507936508
STD: 0.028205773055544364


## Stratified K-Fold Cross Validation

In [13]:
skf=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_model=LogisticRegression(max_iter=10000)
rf_model=RandomForestClassifier(random_state=42)

log_score=cross_val_score(log_model, X, y, cv=skf)
rf_scores=cross_val_score(rf_model, X, y, cv=skf)

print("Logistic Regression Mean:", log_score.mean())
print("Logistic Regression Std:", log_score.std())

print("Random Forest Mean:", rf_scores.mean())
print("Random Forest Std:", rf_scores.std())

Logistic Regression Mean: 0.9663492063492063
Logistic Regression Std: 0.02113317858457236
Random Forest Mean: 0.9774603174603176
Random Forest Std: 0.021299434518521104


## Cross Validation with Multiple Metrics

In [ ]:
scoring=["accuracy", "precision_macro", "recall_macro", "f1_macro"]

log_results=cross_validate(log_model, X, y, cv=skf, scoring=scoring)
rf_results=cross_validate(rf_model, X, y, cv=skf, scoring=scoring)

def print_results(name, results):
    print(f"\n{name}")
    for metric in scoring:
        scores=results[f"test_{metric}"]
        print(f"{metric}: mean={scores.mean():.4f}, std={scores.std():.4f}")
    

print_results("Logistic Regression", log_results)
print_results("Random Forest", rf_results)


Logistic Regression
accuracy: mean=0.9663, std=0.0211
precision_macro: mean=0.9693, std=0.0201
recall_macro: mean=0.9679, std=0.0193
f1_macro: mean=0.9679, std=0.0197

Random Forest
accuracy: mean=0.9775, std=0.0213
precision_macro: mean=0.9776, std=0.0199
recall_macro: mean=0.9802, std=0.0191
f1_macro: mean=0.9784, std=0.0196


## Hyperparameter Tuning (GridSearchCV)

In [17]:
params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 3, 5, 10],
    'min_samples_split': [2, 4, 6]
}

grid=GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Score:", grid.best_score_)
print("Best Params:", grid.best_params_)
print("Best Estimator:", grid.best_estimator_)

Best Score: 0.9785714285714286
Best Params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Best Estimator: RandomForestClassifier(random_state=42)


In [18]:
# Evaluate Best Model
best_model=grid.best_estimator_

y_pred=best_model.predict(X_test)

print("Test Accuracy (Tuned Model):", accuracy_score(y_test, y_pred))

Test Accuracy (Tuned Model): 1.0
